In [42]:
# 1)
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')

columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

df = df[columns].copy()

print(df.isnull().sum())

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [43]:
# 2)
median_hp = df['horsepower'].median()

print("Median horsepower:", median_hp)

Median horsepower: 254.0


In [44]:
# 3)
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()

    return np.sqrt(mse)


# Split data
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


# Target
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']


# ============================
# OPTION 1: FILL WITH ZERO
# ============================

X_train_zero = df_train.fillna(0).values
X_val_zero = df_val.fillna(0).values

w0_zero, w_zero = train_linear_regression(
    X_train_zero,
    y_train
)

y_pred_zero = w0_zero + X_val_zero.dot(w_zero)

rmse_zero = rmse(y_val, y_pred_zero)


# ============================
# OPTION 2: FILL WITH MEAN
# ============================

mean_hp = df_train['horsepower'].mean()

df_train_mean = df_train.copy()
df_val_mean = df_val.copy()

df_train_mean['horsepower'] = (
    df_train_mean['horsepower'].fillna(mean_hp)
)

df_val_mean['horsepower'] = (
    df_val_mean['horsepower'].fillna(mean_hp)
)

X_train_mean = df_train_mean.values
X_val_mean = df_val_mean.values

w0_mean, w_mean = train_linear_regression(
    X_train_mean,
    y_train
)

y_pred_mean = w0_mean + X_val_mean.dot(w_mean)

rmse_mean = rmse(y_val, y_pred_mean)


# Results
zero_score = round(rmse_zero, 3)
mean_score = round(rmse_mean, 3)

print("RMSE with 0:", zero_score)
print("RMSE with mean:", mean_score)

if zero_score < mean_score:
    print("ANSWER: With 0")
elif mean_score < zero_score:
    print("ANSWER: With mean")
else:
    print("ANSWER: Both are equally good")

RMSE with 0: 2.205
RMSE with mean: 2.202
ANSWER: With mean


In [45]:
# 4)
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)

    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


X_train = df_train.fillna(0).values
X_val = df_val.fillna(0).values

r_values = [
    0,
    0.01,
    0.1,
    1,
    5,
    10,
    100
]

results = {}

for r in r_values:

    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r=r
    )

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    rounded_score = round(score, 4)

    results[r] = rounded_score

    print(
        "r =", r,
        "| RMSE =", rounded_score
    )


best_r = min(
    results,
    key=results.get
)

print()
print("ANSWER:", best_r)

r = 0 | RMSE = 2.2053
r = 0.01 | RMSE = 2.2058
r = 0.1 | RMSE = 2.2241
r = 1 | RMSE = 2.3492
r = 5 | RMSE = 2.4094
r = 10 | RMSE = 2.4195
r = 100 | RMSE = 2.4292

ANSWER: 0


In [47]:
# 5)
scores = []

n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test


for seed in range(10):

    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)


    df_train_seed = df.iloc[
        idx[:n_train]
    ].copy()

    df_val_seed = df.iloc[
        idx[n_train:n_train + n_val]
    ].copy()

    df_test_seed = df.iloc[
        idx[n_train + n_val:]
    ].copy()


    df_train_seed = df_train_seed.reset_index(drop=True)
    df_val_seed = df_val_seed.reset_index(drop=True)
    df_test_seed = df_test_seed.reset_index(drop=True)


    y_train_seed = (
        df_train_seed[
            'fuel_efficiency_mpg'
        ].values
    )

    y_val_seed = (
        df_val_seed[
            'fuel_efficiency_mpg'
        ].values
    )


    del df_train_seed['fuel_efficiency_mpg']
    del df_val_seed['fuel_efficiency_mpg']
    del df_test_seed['fuel_efficiency_mpg']


    X_train_seed = (
        df_train_seed
        .fillna(0)
        .values
    )

    X_val_seed = (
        df_val_seed
        .fillna(0)
        .values
    )


    w0, w = train_linear_regression(
        X_train_seed,
        y_train_seed
    )


    y_pred = (
        w0 +
        X_val_seed.dot(w)
    )


    score = rmse(
        y_val_seed,
        y_pred
    )

    scores.append(score)

    print(
        f"Seed {seed}: RMSE = {score}"
    )


std = np.std(scores)

print()
print("RMSE scores:", scores)
print("Standard deviation:", std)
print("ANSWER:", round(std, 3))

Seed 0: RMSE = 2.2392041430552805
Seed 1: RMSE = 2.202112821083947
Seed 2: RMSE = 2.1634197787531013
Seed 3: RMSE = 2.2043588768533255
Seed 4: RMSE = 2.201880094330643
Seed 5: RMSE = 2.2457851509078637
Seed 6: RMSE = 2.269721207950886
Seed 7: RMSE = 2.1907945999367575
Seed 8: RMSE = 2.2166112016940303
Seed 9: RMSE = 2.207036592830149

RMSE scores: [np.float64(2.2392041430552805), np.float64(2.202112821083947), np.float64(2.1634197787531013), np.float64(2.2043588768533255), np.float64(2.201880094330643), np.float64(2.2457851509078637), np.float64(2.269721207950886), np.float64(2.1907945999367575), np.float64(2.2166112016940303), np.float64(2.207036592830149)]
Standard deviation: 0.028781274078160727
ANSWER: 0.029


In [48]:
# 6)
n = len(df)

n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test


np.random.seed(9)

idx = np.arange(n)
np.random.shuffle(idx)


df_train = df.iloc[
    idx[:n_train]
].copy()

df_val = df.iloc[
    idx[n_train:n_train + n_val]
].copy()

df_test = df.iloc[
    idx[n_train + n_val:]
].copy()


df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


# Combine train and validation
df_train_val = pd.concat(
    [
        df_train,
        df_val
    ],
    ignore_index=True
)


# Targets
y_train_val = (
    df_train_val[
        'fuel_efficiency_mpg'
    ].values
)

y_test = (
    df_test[
        'fuel_efficiency_mpg'
    ].values
)


# Remove target
del df_train_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']


# Fill missing values with 0
X_train_val = (
    df_train_val
    .fillna(0)
    .values
)

X_test = (
    df_test
    .fillna(0)
    .values
)


# Train with r = 0.001
w0, w = train_linear_regression_reg(
    X_train_val,
    y_train_val,
    r=0.001
)


# Predict
y_pred_test = (
    w0 +
    X_test.dot(w)
)


# Test RMSE
test_rmse = rmse(
    y_test,
    y_pred_test
)

print("Test RMSE:", test_rmse)
print("ANSWER:", round(test_rmse, 3))

Test RMSE: 2.2358277471948154
ANSWER: 2.236
